# Kernelized polynomial regression

Lecture 4 fits every monomial, one weight per multi-index. The factorial and the Gaussian bump make that inner product the Gaussian kernel

`k(x, x') = exp(-||x - x'||² / 2)`

on the scale the bump was built for. Age is

`f(x*) = k*ᵀ (K + λI)⁻¹ y`

One dual weight per training row. Random features are the same predictor with one ridge weight per cosine feature.

After standardizing ~800 columns, lengthscale 1 makes `K` the identity. This notebook sets the lengthscale to the median squared distance on the training split, and keeps the unit lengthscale as a check.

One 80/20 split of the cleaned training matrix, all columns. `λ` is chosen on an inner 20% of that training split.


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path[:0] = [str(ROOT), str(ROOT / "lib")]

from GLOBAL import RANDOM_STATE
from clean import clean
from load import load


In [2]:

LAMBDAS = np.logspace(-2, 3, 6)
X_all, y_all, _ = load()
X_all, _ = clean(X_all)
y_all = np.ravel(y_all)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=RANDOM_STATE)
scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)
X_te = scaler.transform(X_te)
print(f"train {X_tr.shape}  holdout {X_te.shape}")


Dropping 4 dead columns: ['x104', 'x129', 'x489', 'x530']
Dropping 3 insane-absmax (>1e+10) columns: ['x173', 'x596', 'x665']
train (969, 825)  holdout (243, 825)


In [ ]:
def kernel(A, B, h):
    a2 = np.einsum("ij,ij->i", A, A)[:, None]
    b2 = np.einsum("ij,ij->i", B, B)[None, :]
    sq = np.maximum(a2 + b2 - 2.0 * A @ B.T, 0.0)
    return np.exp(-sq / (2.0 * h))

def median_lengthscale(X, rng):
    n = len(X)
    i = rng.integers(0, n, 4000)
    j = rng.integers(0, n, 4000)
    keep = i != j
    dist = np.sum((X[i[keep]] - X[j[keep]]) ** 2, axis=1)
    return float(np.median(dist))


def predict_kernel(Xtr, ytr, Xte, h, lam):
    K = kernel(Xtr, Xtr, h)
    K.flat[:: K.shape[0] + 1] += lam
    beta = np.linalg.solve(K, ytr)
    return kernel(Xte, Xtr, h) @ beta


def predict_ridge(Xtr, ytr, Xte, lam):
    model = Ridge(alpha=lam)
    model.fit(Xtr, ytr)
    return model.predict(Xte)


def predict_rff(Xtr, ytr, Xte, h, lam, n_components):
    mapped = RBFSampler(gamma=1.0 / (2.0 * h), n_components=n_components, random_state=RANDOM_STATE)
    model = Ridge(alpha=lam)
    model.fit(mapped.fit_transform(Xtr), ytr)
    return model.predict(mapped.transform(Xte))


def choose_lambda(predict, X, y):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    scored = [(lam, r2_score(yb, predict(Xa, ya, Xb, lam))) for lam in LAMBDAS]
    return max(scored, key=lambda item: item[1])


H = median_lengthscale(X_tr, np.random.default_rng(RANDOM_STATE))
print(f"median squared distance {H:.1f}   unit lengthscale 1")


median squared distance 1570.7   unit lengthscale 1


Signature:      
np.cos(
    x,
    /,
    out=None,
    *,
    where=True,
    casting='same_kind',
    order='K',
    dtype=None,
    subok=True,
    signature=None,
)
Call signature:  np.cos(*args, **kwargs)
Type:            ufunc
String form:     <ufunc 'cos'>
File:            ~/code/eth/aml2026-project-1/.venv/lib/python3.13/site-packages/numpy/__init__.py
Docstring:      
cos(x, /, out=None, *, where=True, casting='same_kind', order='K', dtype=None, subok=True[, signature])

Cosine element-wise.

Parameters
----------
x : array_like
    Input array in radians.
out : ndarray, None, or tuple of ndarray and None, optional
    A location into which the result is stored. If provided, it must have
    a shape that the inputs broadcast to. If not provided or None,
    a freshly-allocated array is returned. A tuple (possible only as a
    keyword argument) must have length equal to the number of outputs.
where : array_like, optional
    This condition is broadcast over the input. At loca

In [4]:
import pandas as pd

jobs = {
    "ridge": lambda Xtr, ytr, Xte, lam: predict_ridge(Xtr, ytr, Xte, lam),
    "kernel h=1": lambda Xtr, ytr, Xte, lam: predict_kernel(Xtr, ytr, Xte, 1.0, lam),
    "kernel median h": lambda Xtr, ytr, Xte, lam: predict_kernel(Xtr, ytr, Xte, H, lam),
    "rff 64": lambda Xtr, ytr, Xte, lam: predict_rff(Xtr, ytr, Xte, H, lam, 64),
    "rff 512": lambda Xtr, ytr, Xte, lam: predict_rff(Xtr, ytr, Xte, H, lam, 512),
}

rows = []
for name, predict in jobs.items():
    lam, inner = choose_lambda(predict, X_tr, y_tr)
    pred = predict(X_tr, y_tr, X_te, lam)
    rows.append({
        "model": name,
        "lambda": lam,
        "inner_r2": inner,
        "holdout_r2": r2_score(y_te, pred),
        "holdout_rmse": root_mean_squared_error(y_te, pred),
    })

scores = pd.DataFrame(rows)
print(scores.to_string(index=False, float_format=lambda v: f"{v:.3f}"))


model            lambda  inner_r2  holdout_r2  holdout_rmse
ridge          1000.000     0.289       0.303         7.917
kernel h=1        0.010   -46.899     -54.920        70.933
kernel median h   1.000     0.259       0.349         7.654
rff 64           10.000     0.106       0.154         8.723
rff 512           1.000     0.267       0.290         7.991
